# LeetCode 3560 — Find Minimum Log Transportation Cost bằng Python

Link đề: https://leetcode.com/problems/find-minimum-log-transportation-cost/ · Độ khó: **Easy** · Chủ đề: Math, Greedy

Lưu ý: tên bài trên LeetCode là *Find Minimum Log Transportation Cost*, nhưng hàm cần viết tên là `minCuttingCost` (đúng như skeleton bên dưới).

Skeleton LeetCode:

```python
class Solution:
    def minCuttingCost(self, n: int, m: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có **hai khúc gỗ** dài `n` và `m`, **ba xe tải**, mỗi xe chở **một** khúc dài **không quá `k`**; được **cắt** một khúc thành hai phần, chi phí cắt bằng **tích hai phần**; tìm tổng chi phí **nhỏ nhất** (hoặc `0` nếu không cần cắt).
2. Nhận ra hệ quả của "ba xe, hai khúc": **nhiều nhất một khúc** dài hơn `k` (nếu cả hai dài hơn `k` thì cần 4 xe), và khúc đó chỉ cần **một nhát cắt**.
3. Tối thiểu hoá `a * (x - a)` với ràng buộc `a <= k` và `x - a <= k`: hàm này **lõm** nên nhỏ nhất ở **đầu mút**, cho `k * (x - k)`.
4. Viết 4 cách: vét cạn mọi vị trí cắt, công thức `max`, tách từng khúc (hàm `cost(L)`), và xét hai đầu mút của đoạn cắt hợp lệ; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên nhánh `<= k` (không cần cắt), cắt đôi (cắt cân bằng), lấy `min(n, m)` thay vì `max`, và quên nhân `k`.

## 2. Đề bài & ví dụ

Có hai khúc gỗ dài `n` và `m`, và một robot vận chuyển có **ba xe tải**; mỗi xe chở tối đa **một** khúc gỗ có chiều dài **không vượt quá `k`**. Bạn được **cắt** một khúc gỗ chiều dài `len` thành hai phần `len1` và `len2` (`len1 + len2 = len`, hai phần dương); **chi phí cắt** là `len1 * len2`.

Trả về **tổng chi phí cắt nhỏ nhất** để đưa mọi khúc gỗ lên ba xe tải. Nếu không cần cắt thì trả về `0`.

| Ví dụ | n, m, k | Output | Giải thích |
|---|---|---|---|
| 1 | `6, 5, 5` | 5 | khúc `6 > 5` phải cắt thành `1` và `5`: chi phí `1 * 5 = 5`; ba khúc `1, 5, 5` lên ba xe |
| 2 | `4, 4, 6` | 0 | cả hai khúc `<= 6`, không cần cắt (hai xe dùng, một xe trống) |

Constraints: `2 <= k <= 10^5`; `1 <= n, m <= 2 * k`; đề **đảm bảo** luôn có cách vận chuyển (tức là không thể cả hai khúc đều dài hơn `k`).

In [1]:
n, m, k = 6, 5, 5
longest = max(n, m)
if longest <= k:
    print("không cần cắt: chi phí 0")
else:
    piece = longest - k          # phần còn lại sau khi cắt ra một khúc dài k
    print(f"cắt {longest} thành {k} và {piece}: chi phí {k} * {piece} = {k * piece}")

cắt 6 thành 5 và 1: chi phí 5 * 1 = 5


## 3. Kiến thức nền

### 3.1. Một khúc gỗ cần mấy nhát cắt?

Mỗi xe chở một khúc dài `<= k`. Khúc dài `x` với `k < x <= 2k` chỉ cần **một nhát** cắt thành hai phần `<= k` (mỗi phần cho một xe). Nếu `x > 2k` cần nhiều hơn một nhát và nhiều hơn hai xe — không xảy ra vì `x <= 2 * k` và chỉ có ba xe cho hai khúc.

In [2]:
k = 5
for x in (3, 5, 6, 9, 10):
    pieces = 1 if x <= k else 2
    print(f"x={x}: cần {pieces} xe (cắt {pieces - 1} nhát)")

x=3: cần 1 xe (cắt 0 nhát)
x=5: cần 1 xe (cắt 0 nhát)
x=6: cần 2 xe (cắt 1 nhát)
x=9: cần 2 xe (cắt 1 nhát)
x=10: cần 2 xe (cắt 1 nhát)


### 3.2. Chi phí cắt `a * (x - a)` — hàm bậc hai lõm

Cắt khúc `x` tại vị trí `a` (`1 <= a <= x - 1`) tốn `f(a) = a * (x - a)`. Đây là parabol **lõm** (hệ số `a^2` âm): lớn nhất khi cắt đôi (`a = x / 2`), nhỏ nhất khi cắt **lệch** nhất có thể (đầu mút). Với `x = 6`: `f(1) = 5`, `f(2) = 8`, `f(3) = 9`.

In [3]:
x = 6
print([(a, a * (x - a)) for a in range(1, x)])

[(1, 5), (2, 8), (3, 9), (4, 8), (5, 5)]


### 3.3. Ràng buộc hai phần đều `<= k`

Sau khi cắt, mỗi phần phải `<= k`: `a <= k` và `x - a <= k`, tức `x - k <= a <= k`. Tập `a` hợp lệ là **đoạn** `[x - k, k]` (không rỗng vì `x <= 2k`). Vì `f` lõm, giá trị nhỏ nhất trên đoạn này nằm ở **một trong hai đầu mút**: `a = x - k` hoặc `a = k` — cả hai cho cùng giá trị `k * (x - k)` (do đối xứng).

In [4]:
x, k = 8, 5
valid = [a for a in range(1, x) if a <= k and x - a <= k]
print("a hợp lệ:", valid)
print([(a, a * (x - a)) for a in valid], "-> nhỏ nhất ở hai đầu mút")

a hợp lệ: [3, 4, 5]
[(3, 15), (4, 16), (5, 15)] -> nhỏ nhất ở hai đầu mút


### 3.4. `max`, `min`, và điều kiện ngắn gọn

`max(n, m)` là khúc dài hơn. Câu lệnh điều kiện trong một dòng: `0 if x <= k else k * (x - k)` (biểu thức điều kiện, "ternary"). Chú ý hai khúc có thể đều `<= k` (đáp án `0`) hoặc đúng một khúc `> k`.

In [5]:
n, m, k = 4, 4, 6
x = max(n, m)
print(0 if x <= k else k * (x - k))
n, m, k = 6, 5, 5
x = max(n, m)
print(0 if x <= k else k * (x - k))

0
5


### 3.5. Tính hợp lệ của đầu vào (đảm bảo của đề)

Đề đảm bảo luôn vận chuyển được: tức **không** xảy ra `n > k` và `m > k` cùng lúc (cần 4 xe). Do đó `min(n, m) <= k` luôn đúng. Trong các test của notebook ta sinh ngẫu nhiên và **loại** những bộ `(n, m, k)` vi phạm điều kiện này.

In [6]:
def valid_input(n, m, k):
    return 2 <= k <= 10**5 and 1 <= n <= 2 * k and 1 <= m <= 2 * k and min(n, m) <= k

print(valid_input(6, 5, 5), valid_input(4, 4, 6), valid_input(6, 6, 5))

True True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `n = 6`, `m = 5`, `k = 5`:

```
khúc 6 > 5  -> phải cắt; khúc 5 <= 5 -> không cần cắt
các cách cắt khúc 6:  1+5 (chi phí 5),  2+4 (8),  3+3 (9)    // phần nào cũng <= 5
chọn 1+5: chi phí 5. Ba khúc 1, 5, 5 lên ba xe.
```

Muốn chi phí nhỏ nhất, ta cắt **lệch nhất**: tách ra một khúc dài đúng `k` (đầy xe), phần còn lại `x - k`. Chi phí `k * (x - k)`. Nếu cả hai khúc đều `<= k` thì không cắt gì cả.

## 5. Cách 1 — Brute Force / baseline

Thử **mọi cách**: với mỗi khúc, hoặc **không cắt**, hoặc cắt tại `a = 1..len-1`. Chọn tổ hợp thoả: tổng số khúc `<= 3` và mọi khúc `<= k`; lấy tổng chi phí nhỏ nhất.

Nút thắt: `O((n + m)^2)` khi thử tổ hợp hai khúc (với `n, m` tới `2 * 10^5` là quá chậm). Dùng cho `n, m, k` nhỏ để kiểm chứng.

In [7]:
class SolutionBrute:
    def minCuttingCost(self, n, m, k):
        def options(length):
            # danh sách (số khúc, chi phí, khúc_lớn_nhất) cho mọi cách xử lý một khúc gỗ
            res = [(1, 0, length)]
            for a in range(1, length):
                res.append((2, a * (length - a), max(a, length - a)))
            return res

        best = None
        for pn, cn, mn in options(n):
            for pm, cm, mm in options(m):
                if pn + pm <= 3 and mn <= k and mm <= k:
                    total = cn + cm
                    if best is None or total < best:
                        best = total
        return best

print(SolutionBrute().minCuttingCost(6, 5, 5))
print(SolutionBrute().minCuttingCost(4, 4, 6))

5
0


**Độ phức tạp:** Time `O(n * m)`, Space `O(n + m)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** (1) vì chỉ ba xe cho hai khúc, **nhiều nhất một khúc** dài hơn `k`, đó là `x = max(n, m)`; (2) khúc đó chỉ cần **một nhát**; (3) chi phí `a * (x - a)` lõm nên tối ưu ở đầu mút `a = k` (hoặc `x - k`).

**Quy tắc "nếu… thì…":**

- Nếu `max(n, m) <= k` -> không cần cắt: đáp án `0`.
- Ngược lại -> đáp án `k * (max(n, m) - k)`.

Invariant: khúc ngắn hơn luôn `<= k` (không cần cắt) và cắt khúc dài hơn thành `k` và `x - k` (cả hai `<= k` vì `x <= 2k`).

In [8]:
def formula(n, m, k):
    x = max(n, m)
    return 0 if x <= k else k * (x - k)

print(formula(6, 5, 5), formula(4, 4, 6), formula(10, 3, 5), formula(2, 2, 2))

5 0 25 0


## 7. Các cách tiếp theo

Ta có 4 cách: (1) vét cạn mọi vị trí cắt; (2) công thức theo `max(n, m)`; (3) hàm `cost(L)` cho từng khúc rồi cộng; (4) xét hai đầu mút của đoạn cắt hợp lệ `[x - k, k]`.

### Cách 2 — Công thức theo `max(n, m)`

Một nhánh `if` và một phép nhân, `O(1)`.

In [9]:
class SolutionFormula:
    def minCuttingCost(self, n, m, k):
        x = max(n, m)
        if x <= k:
            return 0
        return k * (x - k)

print(SolutionFormula().minCuttingCost(6, 5, 5))
print(SolutionFormula().minCuttingCost(4, 4, 6))

5
0


Độ phức tạp Cách 2: Time `O(1)`, Space `O(1)`.

### Cách 3 — Hàm `cost(L)` cho từng khúc rồi cộng

Định nghĩa `cost(L) = 0` nếu `L <= k`, ngược lại `k * (L - k)`; đáp án `cost(n) + cost(m)`. Cách này **không cần** gọi `max`, và nhờ đảm bảo "nhiều nhất một khúc dài hơn `k`" nên tổng chỉ có một số hạng khác `0`. Nó cũng tự nhiên khi mở rộng ra nhiều khúc.

In [10]:
class SolutionPerLog:
    def minCuttingCost(self, n, m, k):
        def cost(length):
            return 0 if length <= k else k * (length - k)
        return cost(n) + cost(m)

print(SolutionPerLog().minCuttingCost(6, 5, 5))
print(SolutionPerLog().minCuttingCost(4, 4, 6))

5
0


Độ phức tạp Cách 3: Time `O(1)`, Space `O(1)`.

### Cách 4 — Xét hai đầu mút của đoạn cắt hợp lệ

Với khúc dài `x > k`: đoạn cắt hợp lệ là `a` trong `[x - k, k]`. Vì `f(a) = a * (x - a)` lõm nên `min` của `f` trên đoạn nằm ở `a = x - k` hoặc `a = k`. Tính cả hai và lấy nhỏ hơn (hai giá trị bằng nhau, nhưng viết rõ để thấy lý do).

In [11]:
class SolutionEndpoints:
    def minCuttingCost(self, n, m, k):
        x = max(n, m)
        if x <= k:
            return 0
        lo, hi = x - k, k                 # đoạn cắt hợp lệ: x - k <= a <= k
        f = lambda a: a * (x - a)
        return min(f(lo), f(hi))

print(SolutionEndpoints().minCuttingCost(6, 5, 5))
print(SolutionEndpoints().minCuttingCost(4, 4, 6))

5
0


Độ phức tạp Cách 4: Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 / Cách 4 với `n = 8`, `m = 3`, `k = 5` (khúc `8 > 5`):

| bước | nội dung | giá trị |
|---|---|---|
| 1 | `x = max(8, 3)` | 8 |
| 2 | `x <= k`? (`8 <= 5`) | không -> phải cắt |
| 3 | đoạn cắt hợp lệ `[x - k, k]` | `[3, 5]` |
| 4 | `f(3) = 3 * 5`, `f(4) = 4 * 4`, `f(5) = 5 * 3` | 15, 16, 15 |
| 5 | nhỏ nhất | 15 (đầu mút) |
| 6 | công thức `k * (x - k) = 5 * 3` | 15 |

In [12]:
def trace_log(n, m, k):
    x = max(n, m)
    print(f"x = max({n}, {m}) = {x}; k = {k}")
    if x <= k:
        print("x <= k -> không cần cắt, chi phí 0")
        return 0
    lo, hi = x - k, k
    print(f"đoạn cắt hợp lệ: a trong [{lo}, {hi}]")
    for a in range(lo, hi + 1):
        print(f"  cắt tại a={a}: {a} * {x - a} = {a * (x - a)}")
    ans = k * (x - k)
    print(f"công thức k * (x - k) = {k} * {x - k} = {ans}")
    return ans

trace_log(8, 3, 5)
trace_log(6, 5, 5)
trace_log(4, 4, 6)

x = max(8, 3) = 8; k = 5
đoạn cắt hợp lệ: a trong [3, 5]
  cắt tại a=3: 3 * 5 = 15
  cắt tại a=4: 4 * 4 = 16
  cắt tại a=5: 5 * 3 = 15
công thức k * (x - k) = 5 * 3 = 15
x = max(6, 5) = 6; k = 5
đoạn cắt hợp lệ: a trong [1, 5]
  cắt tại a=1: 1 * 5 = 5
  cắt tại a=2: 2 * 4 = 8
  cắt tại a=3: 3 * 3 = 9
  cắt tại a=4: 4 * 2 = 8
  cắt tại a=5: 5 * 1 = 5
công thức k * (x - k) = 5 * 1 = 5
x = max(4, 4) = 4; k = 6
x <= k -> không cần cắt, chi phí 0


0

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minCuttingCost(self, n: int, m: int, k: int) -> int:
        x = max(n, m)
        if x <= k:
            return 0
        return k * (x - k)

print(Solution().minCuttingCost(6, 5, 5), Solution().minCuttingCost(4, 4, 6))

5 0


## 10. Vì sao thuật toán đúng

- **Nhiều nhất một khúc cần cắt:** ba xe, mỗi xe một khúc. Nếu cả `n > k` và `m > k` thì cả hai đều phải cắt (mỗi khúc cần `>= 2` xe) -> `>= 4` xe, vô lý. Đề đảm bảo điều này không xảy ra, nên khúc ngắn hơn `<= k` và đi thẳng.
- **Khúc dài `x` chỉ cần một nhát:** `x <= 2k` nên cắt thành hai phần `<= k` là đủ, dùng đúng 2 xe; tổng xe `<= 3` (khúc kia 1 xe). Cắt thêm nhát nào nữa chỉ tốn thêm (chi phí dương) và dùng thêm xe.
- **Vị trí cắt tối ưu:** phần `a` hợp lệ khi `x - k <= a <= k`. `f(a) = a * (x - a) = -a^2 + x * a` là parabol lõm nên giá trị nhỏ nhất trên đoạn nằm ở **đầu mút**: `f(x - k) = (x - k) * k` và `f(k) = k * (x - k)`, bằng nhau. Đáp án `k * (x - k)`.
- **Không cần cắt:** nếu `x <= k` thì cả hai khúc đã lên được xe, chi phí `0` (chi phí không âm nên `0` là nhỏ nhất).
- **Cách 3** cộng chi phí từng khúc: khúc `<= k` cho `0`, khúc `> k` cho `k * (L - k)`; vì tối đa một khúc `> k` nên tổng bằng đáp án của Cách 2.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi vị trí cắt | `O(n * m)` | `O(n + m)` |
| 2. Công thức `max` | `O(1)` | `O(1)` |
| 3. Hàm `cost(L)` từng khúc | `O(1)` | `O(1)` |
| 4. Hai đầu mút | `O(1)` | `O(1)` |

Cách 2, 3, 4 đều `O(1)`. Với `n, m` tới `2 * 10^5`, tích `k * (x - k)` tối đa `10^5 * 10^5 = 10^10` — **vượt** `int` 32 bit (`~2.1 * 10^9`): ở C++/Java cần `long long`. Python không tràn.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (không cần cắt, đúng bằng `k`, vừa vượt `k`, đúng `2k`, `k` nhỏ nhất, biên lớn `10^5`).

In [14]:
impls = [SolutionBrute(), SolutionFormula(), SolutionPerLog(), SolutionEndpoints(), Solution()]

def valid_input(n, m, k):
    return 2 <= k <= 10**5 and 1 <= n <= 2 * k and 1 <= m <= 2 * k and min(n, m) <= k

tests = [
    (6, 5, 5, 5),
    (4, 4, 6, 0),
    (5, 5, 5, 0),                     # đúng bằng k: không cần cắt
    (6, 1, 5, 5),                     # vừa vượt k một đơn vị: 5 * 1
    (10, 1, 5, 25),                   # đúng 2k: cắt đôi k + k, 5 * 5 (cắt lệch hơn không hợp lệ)
    (1, 10, 5, 25),                   # đổi chỗ n, m
    (2, 2, 2, 0),                     # k nhỏ nhất
    (4, 1, 2, 4),                     # 4 = 2k, cắt 2 + 2: 2 * 2
    (3, 1, 2, 2),                     # 3 -> 2 + 1: 2 * 1
    (8, 3, 5, 15),                    # ví dụ ở bảng trace
    (10**5, 10**5, 10**5, 0),         # biên lớn, không cần cắt
    (2 * 10**5, 1, 10**5, 10**10),    # biên lớn, cắt đôi k + k: 10^5 * 10^5
    (10**5 + 1, 10**5, 10**5, 10**5), # vượt đúng 1 ở biên lớn
]
for n, m, k, exp in tests:
    assert valid_input(n, m, k), (n, m, k)
    for im in impls:
        if isinstance(im, SolutionBrute) and n + m > 100:
            continue                  # brute O(n*m): bỏ ca rất lớn
        got = im.minCuttingCost(n, m, k)
        assert got == exp, (type(im).__name__, n, m, k, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed, đúng constraints và **đúng đảm bảo của đề** (bỏ các bộ mà cả hai khúc `> k`). Với `k` nhỏ đối chiếu **cả brute force**; với `k` lớn đối chiếu các cách `O(1)`.

In [15]:
import random
random.seed(3560)

cnt = 0
while cnt < 1500:
    k = random.randint(2, 20)
    n = random.randint(1, 2 * k)
    m = random.randint(1, 2 * k)
    if not valid_input(n, m, k):
        continue
    cnt += 1
    exp = impls[0].minCuttingCost(n, m, k)
    for im in impls:
        assert im.minCuttingCost(n, m, k) == exp, (type(im).__name__, n, m, k)
print("1500 test nhỏ (có brute) OK")

cnt = 0
while cnt < 2000:
    k = random.randint(2, 10**5)
    n = random.randint(1, 2 * k)
    m = random.randint(1, 2 * k)
    if not valid_input(n, m, k):
        continue
    cnt += 1
    exp = SolutionFormula().minCuttingCost(n, m, k)
    for im in impls[2:]:
        assert im.minCuttingCost(n, m, k) == exp, (type(im).__name__, n, m, k)
print("2000 test lớn OK")

1500 test nhỏ (có brute) OK
2000 test lớn OK


Test 3: tính chất — đối xứng `f(n, m, k) == f(m, n, k)`; đáp án `>= 0`; bằng `0` khi và chỉ khi `max(n, m) <= k`; tăng (không giảm) khi khúc dài tăng; không vượt `k * k` (vì `x - k <= k`).

In [16]:
random.seed(560)
cnt = 0
while cnt < 1000:
    k = random.randint(2, 10**5)
    n = random.randint(1, 2 * k)
    m = random.randint(1, 2 * k)
    if not valid_input(n, m, k):
        continue
    cnt += 1
    ans = Solution().minCuttingCost(n, m, k)
    assert ans == Solution().minCuttingCost(m, n, k)
    assert ans >= 0
    assert (ans == 0) == (max(n, m) <= k)
    assert ans <= k * k
    n2 = min(n + 1, 2 * k)
    if valid_input(n2, m, k):
        assert Solution().minCuttingCost(n2, m, k) >= ans
print("tính chất (đối xứng, >= 0, = 0 khi max <= k, <= k*k, đơn điệu): OK")

tính chất (đối xứng, >= 0, = 0 khi max <= k, <= k*k, đơn điệu): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên nhánh `max(n, m) <= k` (không cần cắt)

In [17]:
def wrong_always_cut(n, m, k):
    x = max(n, m)
    return k * (x - k)                          # SAI: khi x <= k thì x - k <= 0, ra số âm / vô nghĩa

print("sai :", wrong_always_cut(4, 4, 6))
print("đúng:", Solution().minCuttingCost(4, 4, 6))
assert wrong_always_cut(4, 4, 6) != Solution().minCuttingCost(4, 4, 6)

sai : -12
đúng: 0


Vì sao sai: `6 * (4 - 6) = -12` — chi phí âm là vô lý; thực tế không phải cắt nên chi phí `0`. **Sửa:** `if x <= k: return 0`.

### Lỗi 2: cắt đôi (cắt cân bằng) cho công bằng

In [18]:
def wrong_halve(n, m, k):
    x = max(n, m)
    if x <= k:
        return 0
    a = x // 2
    return a * (x - a)                          # SAI: cắt đôi cho chi phí LỚN nhất, không phải nhỏ nhất

print("sai :", wrong_halve(6, 5, 5))
print("đúng:", Solution().minCuttingCost(6, 5, 5))
assert wrong_halve(6, 5, 5) != Solution().minCuttingCost(6, 5, 5)

sai : 9
đúng: 5


Vì sao sai: tích `a * (x - a)` lớn nhất khi hai phần bằng nhau. Ta muốn **nhỏ nhất** nên cắt **lệch** nhất có thể (tách ra phần dài đúng `k`). **Sửa:** `k * (x - k)`.

### Lỗi 3: lấy `min(n, m)` thay vì `max(n, m)`

In [19]:
def wrong_min_log(n, m, k):
    x = min(n, m)                               # SAI: khúc cần cắt là khúc DÀI hơn
    if x <= k:
        return 0
    return k * (x - k)

print("sai :", wrong_min_log(6, 5, 5))
print("đúng:", Solution().minCuttingCost(6, 5, 5))
assert wrong_min_log(6, 5, 5) != Solution().minCuttingCost(6, 5, 5)

sai : 0
đúng: 5


Vì sao sai: `min(6, 5) = 5 <= 5` nên trả `0`, bỏ qua khúc `6` vẫn dài hơn `k = 5`. **Sửa:** xét khúc **dài nhất**.

### Lỗi 4: quên nhân `k` (chi phí là tích, không phải độ dài thừa)

In [20]:
def wrong_no_multiply(n, m, k):
    x = max(n, m)
    if x <= k:
        return 0
    return x - k                                # SAI: chi phí là TÍCH hai phần, không phải phần dư

print("sai :", wrong_no_multiply(6, 5, 5))
print("đúng:", Solution().minCuttingCost(6, 5, 5))
assert wrong_no_multiply(6, 5, 5) != Solution().minCuttingCost(6, 5, 5)

sai : 1
đúng: 5


Vì sao sai: cắt `6` thành `1` và `5` tốn `1 * 5 = 5`, không phải `1`. Luôn đọc kỹ định nghĩa chi phí cắt: `len1 * len2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đọc **đảm bảo của đề** ("luôn vận chuyển được") để suy ra cấu trúc: ở đây tối đa một khúc dài hơn `k`.
- "Chi phí = tích hai phần, tối thiểu hoá" -> cắt **lệch** nhất có thể; nếu "tối đa hoá" -> cắt đôi.
- Hàm bậc hai lõm/lồi trên một đoạn: cực trị ở **đầu mút** (lõm -> min ở đầu mút; lồi -> min ở đỉnh).
- Luôn có nhánh "không cần làm gì" (`<= k` -> `0`) trước công thức.
- Kiểm tra bằng **brute force** trên `k` nhỏ để chắc công thức.
- Giá trị tới `10^10` -> cẩn thận tràn số ở C++/Java.

## 15. Biến thể của bài

1. **Nhiều khúc gỗ** và nhiều xe: tổng số nhát cắt cần thiết, mỗi khúc `> k` cần `ceil(L / k) - 1` nhát.
2. **Chi phí cắt khác**: `len1 + len2` hay `max(len1, len2)` (đổi hàm mục tiêu, cực trị khác).
3. Mỗi xe chở **tổng** các khúc `<= k` (bin packing) chứ không chỉ một khúc.
4. Khúc gỗ dài tới `3k` hoặc hơn: cần **nhiều nhát**; tối ưu bằng cách cắt từng đoạn dài `k` từ một đầu.
5. **Tối đa hoá** chi phí (cắt đôi): `(x // 2) * (x - x // 2)`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn vị trí cắt | thử mọi `a` cho cả hai khúc | `O(n * m)` | `O(n + m)` | kiểm chứng, `k` nhỏ |
| 2. Công thức `max` | `k * (max - k)` hoặc `0` | `O(1)` | `O(1)` | **chuẩn, nên nộp** |
| 3. `cost(L)` từng khúc | cộng chi phí hai khúc | `O(1)` | `O(1)` | mở rộng nhiều khúc |
| 4. Hai đầu mút | `min(f(x - k), f(k))` | `O(1)` | `O(1)` | hiểu vì sao cắt lệch |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minCuttingCost(9, 4, 6)` bằng bao nhiêu?

**Bài 2 (tính tay):** bộ `(n, m, k) = (7, 7, 5)` có hợp lệ theo đảm bảo của đề không (hai khúc đều `> k`)? Sau đó tính đáp án cho `(7, 3, 5)`.

**Bài 3 (điền TODO):** hoàn thành `cost_one(length, k)` trả về chi phí cắt nhỏ nhất cho **một** khúc gỗ (`0` nếu `<= k`).

**Bài 4 (tự viết lại):** viết `cut_plan(n, m, k)` trả về danh sách các khúc sau khi cắt tối ưu (ví dụ `(6, 5, 5)` -> `[1, 5, 5]`, khúc `<= k` giữ nguyên).

**Bài 5 (mini challenge):** viết `max_cutting_cost(n, m, k)` — chi phí **lớn nhất** (cắt đôi khúc dài nếu cần) và kiểm tra `>=` đáp án nhỏ nhất.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minCuttingCost(9, 4, 6))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def cost_one(length, k):
    # TODO: 0 nếu length <= k, ngược lại k * (length - k)
    return -1

18


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** chỉ khúc dài hơn `k` mới phải cắt; cắt sao cho một phần dài đúng `k` thì chi phí nhỏ nhất.

**Mức 2 (vừa):** Bài 3: một nhánh `if` và một phép nhân. Bài 4: với `x > k` thay `x` bằng `[x - k, k]`. Bài 5: cắt đôi `a = x // 2`, chi phí `a * (x - a)`, vẫn cần `a <= k` và `x - a <= k` (đúng vì `x <= 2k`).

**Mức 3 (gần đáp án):** Bài 1: khúc `9 > 6`, `6 * (9 - 6) = 18`. Bài 2: `(7, 7, 5)` vi phạm đảm bảo (cả hai `> 5`); `(7, 3, 5)`: `5 * 2 = 10`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: 6 * (9 - 6) = 18
assert Solution().minCuttingCost(9, 4, 6) == 18

# Bài 2: (7, 7, 5) không hợp lệ; (7, 3, 5) -> 5 * 2 = 10
assert not valid_input(7, 7, 5)
assert valid_input(7, 3, 5)
assert Solution().minCuttingCost(7, 3, 5) == 10

# Bài 3
def cost_one(length, k):
    return 0 if length <= k else k * (length - k)
assert cost_one(6, 5) == 5
assert cost_one(4, 6) == 0
assert cost_one(10, 5) == 25

# Bài 4
def cut_plan(n, m, k):
    pieces = []
    for L in (n, m):
        if L <= k:
            pieces.append(L)
        else:
            pieces.extend([L - k, k])
    return sorted(pieces)
assert cut_plan(6, 5, 5) == [1, 5, 5]
assert cut_plan(4, 4, 6) == [4, 4]
assert len(cut_plan(10, 3, 5)) <= 3
assert all(p <= 5 for p in cut_plan(10, 3, 5))

# Bài 5
def max_cutting_cost(n, m, k):
    x = max(n, m)
    if x <= k:
        return 0
    a = x // 2
    return a * (x - a)
assert max_cutting_cost(6, 5, 5) == 9
assert max_cutting_cost(4, 4, 6) == 0
for (n, m, k) in ((6, 5, 5), (8, 3, 5), (10, 1, 5), (4, 4, 6)):
    assert max_cutting_cost(n, m, k) >= Solution().minCuttingCost(n, m, k)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cắt lệch nhất để tối thiểu hoá tích":**

```python
x = max(n, m)           # khúc dài nhất (tối đa một khúc > k theo đảm bảo của đề)
if x <= k:
    return 0            # không cần cắt
return k * (x - k)      # cắt ra một khúc đúng k, phần còn lại x - k
```

**Checklist trước khi submit:**

- [ ] Có nhánh `x <= k` trả `0` trước khi dùng công thức?
- [ ] Dùng `max(n, m)` (khúc dài nhất), không phải `min`?
- [ ] Cắt **lệch** (một phần đúng `k`), không cắt đôi?
- [ ] Chi phí là **tích** `k * (x - k)`, không phải hiệu?
- [ ] Hiểu đảm bảo của đề (tối đa một khúc `> k`, nên một nhát cắt)?
- [ ] Số lớn `10^10` -> đã nghĩ tới tràn số ở ngôn ngữ khác?
- [ ] Chữ ký đúng `minCuttingCost(self, n: int, m: int, k: int) -> int`?